# Notebook 04 — Hybrid Retrieval + RRF

**Goal:** combine dense semantic retrieval from FAISS with lexical retrieval from BM25, then fuse both rankings using Reciprocal Rank Fusion (RRF).

```text
User Query
    │
    ├──► Dense Retrieval ──► FAISS
    │
    └──► Lexical Retrieval ─► BM25
                │
                ▼
        Hybrid Retrieval
                │
                ▼
               RRF
                │
                ▼
        Top-K Candidate Chunks
```

**Inputs**
- `quant_research.faiss`
- `quant_research_metadata.json`

**Outputs**
- persisted BM25 index
- BM25 chunk mapping
- dense retrieval function
- BM25 retrieval function
- RRF fusion
- hybrid retrieval function


In [1]:
# 01 — Imports and configuration

import json
import pickle
import re
from pathlib import Path

import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

DATA_DIR = Path("../data")
VECTOR_STORE_DIR = DATA_DIR / "vector_store"

INDEX_PATH = VECTOR_STORE_DIR / "quant_research.faiss"
METADATA_PATH = VECTOR_STORE_DIR / "quant_research_metadata.json"

BM25_PATH = VECTOR_STORE_DIR / "quant_research_bm25.pkl"
BM25_IDS_PATH = VECTOR_STORE_DIR / "quant_research_bm25_ids.json"
HYBRID_CONFIG_PATH = VECTOR_STORE_DIR / "hybrid_retrieval_config.json"

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

DENSE_TOP_K = 20
BM25_TOP_K = 20
FINAL_TOP_K = 10
RRF_K = 60

print("FAISS:", INDEX_PATH)
print("Metadata:", METADATA_PATH)
print("BM25:", BM25_PATH)


c:\Users\MD RAJA\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FAISS: ..\data\vector_store\quant_research.faiss
Metadata: ..\data\vector_store\quant_research_metadata.json
BM25: ..\data\vector_store\quant_research_bm25.pkl


In [2]:
# 02 — Load FAISS index and metadata

assert INDEX_PATH.exists(), f"Missing FAISS index: {INDEX_PATH}"
assert METADATA_PATH.exists(), f"Missing metadata: {METADATA_PATH}"

dense_index = faiss.read_index(str(INDEX_PATH))

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("FAISS vectors:", dense_index.ntotal)
print("Metadata records:", len(metadata))

assert dense_index.ntotal == len(metadata)

chunk_ids = [m["chunk_id"] for m in metadata]
assert len(chunk_ids) == len(set(chunk_ids))

print("Unique chunk IDs:", len(set(chunk_ids)))


FAISS vectors: 1141
Metadata records: 1141
Unique chunk IDs: 1141


In [3]:
# 03 — Prepare lexical corpus for BM25

def tokenize_for_bm25(text):
    text = str(text).lower()
    return re.findall(r"(?u)\b\w+\b", text)

bm25_records = []
bm25_tokenized_corpus = []
bm25_chunk_ids = []

for record in metadata:
    text = record.get("retrieval_text") or record.get("text") or ""
    text = text.strip()

    # Evidence-only chunks without text remain in metadata/FAISS,
    # but are excluded from lexical BM25 retrieval.
    if not text:
        continue

    tokens = tokenize_for_bm25(text)

    if not tokens:
        continue

    bm25_records.append(record)
    bm25_tokenized_corpus.append(tokens)
    bm25_chunk_ids.append(record["chunk_id"])

print("Total metadata records:", len(metadata))
print("BM25 records:", len(bm25_records))
print("Excluded text-empty records:", len(metadata) - len(bm25_records))

assert len(bm25_records) == len(bm25_tokenized_corpus)
assert len(bm25_records) == len(bm25_chunk_ids)
assert len(bm25_chunk_ids) == len(set(bm25_chunk_ids))


Total metadata records: 1141
BM25 records: 1132
Excluded text-empty records: 9


In [4]:
# 04 — Build and persist BM25 index

bm25 = BM25Okapi(bm25_tokenized_corpus)

with open(BM25_PATH, "wb") as f:
    pickle.dump(bm25, f)

with open(BM25_IDS_PATH, "w", encoding="utf-8") as f:
    json.dump(bm25_chunk_ids, f, indent=2)

with open(HYBRID_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "embedding_model": EMBEDDING_MODEL_NAME,
            "dense_top_k": DENSE_TOP_K,
            "bm25_top_k": BM25_TOP_K,
            "final_top_k": FINAL_TOP_K,
            "rrf_k": RRF_K,
            "bm25_tokenizer": "lowercase_word_regex",
        },
        f,
        indent=2,
    )

print("BM25 index saved:", BM25_PATH)
print("BM25 chunk mapping saved:", BM25_IDS_PATH)
print("Hybrid config saved:", HYBRID_CONFIG_PATH)


BM25 index saved: ..\data\vector_store\quant_research_bm25.pkl
BM25 chunk mapping saved: ..\data\vector_store\quant_research_bm25_ids.json
Hybrid config saved: ..\data\vector_store\hybrid_retrieval_config.json


In [5]:
# 05 — Reload BM25 artifacts and verify persistence

with open(BM25_PATH, "rb") as f:
    loaded_bm25 = pickle.load(f)

with open(BM25_IDS_PATH, "r", encoding="utf-8") as f:
    loaded_bm25_ids = json.load(f)

print("Reloaded BM25 records:", len(loaded_bm25_ids))

assert len(loaded_bm25_ids) == len(bm25_records)
assert loaded_bm25_ids == bm25_chunk_ids

print("BM25 persistence check: PASS")


Reloaded BM25 records: 1132
BM25 persistence check: PASS


In [6]:
# 06 — Load embedding model for query encoding

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2926.60it/s]


Embedding model: BAAI/bge-small-en-v1.5
Embedding dimension: 384


C:\Users\MD RAJA\AppData\Local\Temp\ipykernel_26572\2797710805.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


In [7]:
# 07 — Dense retrieval

def dense_retrieve(query, top_k=DENSE_TOP_K):
    query_vector = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, indices = dense_index.search(
        query_vector,
        min(top_k, dense_index.ntotal),
    )

    results = []

    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        if idx < 0:
            continue

        record = metadata[int(idx)]

        results.append({
            "chunk_id": record["chunk_id"],
            "score": float(score),
            "rank": rank,
            "source_file": record.get("source_file"),
            "section_path": record.get("section_path"),
            "pages": record.get("pages"),
            "text": record.get("retrieval_text") or record.get("text") or "",
        })

    return results


In [8]:
# 08 — BM25 lexical retrieval

def bm25_retrieve(query, top_k=BM25_TOP_K):
    query_tokens = tokenize_for_bm25(query)

    if not query_tokens:
        return []

    scores = loaded_bm25.get_scores(query_tokens)

    ranked_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True,
    )[:top_k]

    results = []

    for rank, idx in enumerate(ranked_indices, start=1):
        record = bm25_records[idx]

        results.append({
            "chunk_id": record["chunk_id"],
            "score": float(scores[idx]),
            "rank": rank,
            "source_file": record.get("source_file"),
            "section_path": record.get("section_path"),
            "pages": record.get("pages"),
            "text": record.get("retrieval_text") or record.get("text") or "",
        })

    return results


In [9]:
# 09 — Reciprocal Rank Fusion (RRF)

def reciprocal_rank_fusion(
    dense_results,
    bm25_results,
    top_k=FINAL_TOP_K,
    rrf_k=RRF_K,
):
    # RRF(d) = sum(1 / (k + rank))
    fused = {}

    for result in dense_results:
        chunk_id = result["chunk_id"]

        fused.setdefault(
            chunk_id,
            {
                "chunk_id": chunk_id,
                "rrf_score": 0.0,
                "dense_rank": None,
                "bm25_rank": None,
            },
        )

        fused[chunk_id]["dense_rank"] = result["rank"]
        fused[chunk_id]["rrf_score"] += (
            1.0 / (rrf_k + result["rank"])
        )

    for result in bm25_results:
        chunk_id = result["chunk_id"]

        fused.setdefault(
            chunk_id,
            {
                "chunk_id": chunk_id,
                "rrf_score": 0.0,
                "dense_rank": None,
                "bm25_rank": None,
            },
        )

        fused[chunk_id]["bm25_rank"] = result["rank"]
        fused[chunk_id]["rrf_score"] += (
            1.0 / (rrf_k + result["rank"])
        )

    metadata_by_id = {m["chunk_id"]: m for m in metadata}

    ranked = sorted(
        fused.values(),
        key=lambda x: x["rrf_score"],
        reverse=True,
    )

    final_results = []

    for rank, item in enumerate(ranked[:top_k], start=1):
        record = metadata_by_id[item["chunk_id"]]

        final_results.append({
            "rank": rank,
            "chunk_id": item["chunk_id"],
            "rrf_score": item["rrf_score"],
            "dense_rank": item["dense_rank"],
            "bm25_rank": item["bm25_rank"],
            "source_file": record.get("source_file"),
            "section_path": record.get("section_path"),
            "pages": record.get("pages"),
            "element_ids": record.get("element_ids", []),
            "content_types": record.get("content_types", []),
            "image_paths": record.get("image_paths", []),
            "evidence": record.get("evidence", []),
            "text": record.get("retrieval_text") or record.get("text") or "",
        })

    return final_results


In [10]:
# 10 — Hybrid retrieval

def hybrid_retrieve(
    query,
    dense_top_k=DENSE_TOP_K,
    bm25_top_k=BM25_TOP_K,
    final_top_k=FINAL_TOP_K,
):
    dense_results = dense_retrieve(query, dense_top_k)
    bm25_results = bm25_retrieve(query, bm25_top_k)

    hybrid_results = reciprocal_rank_fusion(
        dense_results,
        bm25_results,
        top_k=final_top_k,
        rrf_k=RRF_K,
    )

    return {
        "query": query,
        "dense_results": dense_results,
        "bm25_results": bm25_results,
        "hybrid_results": hybrid_results,
    }


query = "How does portfolio optimization balance expected return and risk?"

hybrid_output = hybrid_retrieve(query)

print("Query:", query)
print("Dense candidates:", len(hybrid_output["dense_results"]))
print("BM25 candidates:", len(hybrid_output["bm25_results"]))
print("Final hybrid results:", len(hybrid_output["hybrid_results"]))


Query: How does portfolio optimization balance expected return and risk?
Dense candidates: 20
BM25 candidates: 20
Final hybrid results: 10


In [11]:
# 11 — Inspect hybrid results

for result in hybrid_output["hybrid_results"]:
    print("=" * 100)
    print("Rank:", result["rank"])
    print("Chunk:", result["chunk_id"])
    print("RRF score:", round(result["rrf_score"], 6))
    print("Dense rank:", result["dense_rank"])
    print("BM25 rank:", result["bm25_rank"])
    print("Source:", result["source_file"])
    print("Pages:", result["pages"])
    print("Section:", result["section_path"])
    print("Content types:", result["content_types"])
    print("Text:", result["text"][:600])


Rank: 1
Chunk: chunk_6a3e8d2924b52017
RRF score: 0.032266
Dense rank: 3
BM25 rank: 1
Source: 2605.29413v1.pdf
Pages: [6, 7]
Section: ['4.1 Unconstrained MVO and the Corner Portfolio Problem']
Content types: ['text']
Text: The initial optimization framework implemented in this study solves for the Global Minimum Variance (GMV) a portfolio which is not necessarily the tangent portfolio. Whereas the tangent portfolio gives the Sharpe maximising portfolio as it simultaneously minimizes the volatility and maximizes the expected return, the optimization problem solved in this work does not take the expected return into consideration but rather focuses on the variance minimization as long as it satisfies certain requirements. Formally, the optimization problem is specified as: subject to:
Rank: 2
Chunk: chunk_3242642c4df23aa2
RRF score: 0.032018
Dense rank: 1
BM25 rank: 4
Source: 2605.28853v1.pdf
Pages: [2]
Section: ['A. Problem Statement and Preliminaries']
Content types: ['text']
Text: Port

In [12]:
# 12 — Retrieval QA

metadata_ids = {m["chunk_id"] for m in metadata}

assert dense_index.ntotal == len(metadata)
assert len(bm25_chunk_ids) > 0
assert len(bm25_chunk_ids) == len(set(bm25_chunk_ids))

for result in hybrid_output["hybrid_results"]:
    assert result["chunk_id"] in metadata_ids
    assert result["rank"] >= 1
    assert result["rrf_score"] > 0

print("Hybrid retrieval QA: PASS")
print("Dense index records:", dense_index.ntotal)
print("BM25 records:", len(bm25_chunk_ids))
print("Hybrid top-K:", len(hybrid_output["hybrid_results"]))


Hybrid retrieval QA: PASS
Dense index records: 1141
BM25 records: 1132
Hybrid top-K: 10


## Notebook 04 → Notebook 05 contract

Notebook 05 will consume the hybrid candidate set:

```text
FAISS Dense Retrieval
        +
BM25 Lexical Retrieval
        ↓
       RRF
        ↓
Hybrid Candidate Set
        ↓
Cross-Encoder Reranking
```

The following provenance remains attached to every hybrid result:

- `chunk_id`
- `source_file`
- `section_path`
- `pages`
- `element_ids`
- `content_types`
- `image_paths`
- `evidence`

This preserves the connection from retrieval → original PDF evidence → later citations and traceability.
